<a href="https://colab.research.google.com/github/suparuek2405/GemmaSQL/blob/main/GemmaSQL(Github).ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

### Install requirement

In [ ]:

%%capture
import os, re
if "COLAB_" not in "".join(os.environ.keys()):
    !pip install unsloth
else:
    # Do this only in Colab notebooks! Otherwise use pip install unsloth
    import torch; v = re.match(r"[0-9\.]{3,}", str(torch.__version__)).group(0)
    xformers = "xformers==" + ("0.0.32.post2" if v == "2.8.0" else "0.0.29.post3")
    !pip install --no-deps bitsandbytes accelerate {xformers} peft trl triton cut_cross_entropy unsloth_zoo
    !pip install sentencepiece protobuf "datasets>=3.4.1,<4.0.0" "huggingface_hub>=0.34.0" hf_transfer
    !pip install --no-deps unsloth
!pip install transformers==4.56.2
!pip install --no-deps trl==0.22.2


In [ ]:
%%capture
!pip install --no-deps --upgrade timm # Only for Gemma 3N

In [ ]:
from huggingface_hub import login, whoami
your_token = "xxxxxx"
login(token=your_token, add_to_git_credential=True)

print(whoami())

{'type': 'user', 'id': '672791aafab7cbfa931c6757', 'name': 'suparuek2405', 'fullname': 'suparuek', 'email': 'suparuek2405@gmail.com', 'emailVerified': True, 'canPay': False, 'billingMode': 'prepaid', 'periodEnd': 1772323200, 'isPro': False, 'avatarUrl': 'https://cdn-avatars.huggingface.co/v1/production/uploads/no-auth/6FjibcVKjHN3v0JhwSLHE.png', 'orgs': [], 'auth': {'type': 'access_token', 'accessToken': {'displayName': 'forbooksql_access_write', 'role': 'write', 'createdAt': '2025-10-05T14:22:48.214Z'}}}


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


### Load model / Load trained adapter

#### Load original model (Skip this)

In [ ]:
# Load model
from unsloth import FastModel
import torch

model, tokenizer = FastModel.from_pretrained(
    model_name = "unsloth/gemma-3n-E4B-it",
    dtype = None, # None for auto detection
    max_seq_length = 4096, # Choose any for long context!
    load_in_4bit = True,  # 4 bit quantization to reduce memory
    full_finetuning = False, # [NEW!] We have full finetuning now!
    # token = "hf_...", # use one if using gated models
)

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
Unsloth: Could not import trl.trainer.alignprop_trainer: Failed to import trl.trainer.alignprop_trainer because of the following error (look up to see its traceback):
Failed to import trl.models.modeling_sd_base because of the following error (look up to see its traceback):
Failed to import diffusers.pipelines.stable_diffusion.pipeline_stable_diffusion because of the following error (look up to see its traceback):
Failed to import diffusers.loaders.ip_adapter because of the following error (look up to see its traceback):
JITCallable._set_src() takes 1 positional argument but 2 were given
Unsloth: Could not import trl.trainer.ddpo_trainer: Failed to import trl.trainer.ddpo_trainer because of the following error (look up to see its traceback):
Failed to import trl.models.modeling_sd_base because of the following error (look up to see its traceback):
Fa

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 3 files:   0%|          | 0/3 [00:00<?, ?it/s]

model-00001-of-00003.safetensors:   0%|          | 0.00/3.72G [00:00<?, ?B/s]

model-00003-of-00003.safetensors:   0%|          | 0.00/1.15G [00:00<?, ?B/s]

...

preprocessor_config.json: 0.00B [00:00, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.model:   0%|          | 0.00/4.70M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/33.4M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/777 [00:00<?, ?B/s]

#### Load Finetuned model

In [ ]:
from unsloth import FastModel
from transformers import AutoTokenizer
from peft import PeftModel

your_adapter_repo = 'xxxxx'

BASE_MODEL   = "unsloth/gemma-3n-E4B-it"
ADAPTER_REPO = your_adapter_repo

USE_AUTH     = True

base_model, _ = FastModel.from_pretrained(
    model_name     = BASE_MODEL,
    max_seq_length = 4096,
    dtype          = None,
    load_in_4bit   = True,
    full_finetuning= False,
)

tokenizer = AutoTokenizer.from_pretrained(ADAPTER_REPO, use_auth_token=USE_AUTH)

model = PeftModel.from_pretrained(
    base_model, ADAPTER_REPO,
    is_trainable=True,
    use_auth_token=USE_AUTH,
)

model.train()
model.print_trainable_parameters()

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
==((====))==  Unsloth 2026.1.4: Fast Gemma3N patching. Transformers: 4.56.2.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.741 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.9.0+cu126. CUDA: 7.5. CUDA Toolkit: 12.6. Triton: 3.5.0
\        /    Bfloat16 = FALSE. FA [Xformers = None. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!
Unsloth: Using float16 precision for gemma3n won't work! Using float32.


model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 3 files:   0%|          | 0/3 [00:00<?, ?it/s]

model-00003-of-00003.safetensors:   0%|          | 0.00/1.15G [00:00<?, ?B/s]

model-00001-of-00003.safetensors:   0%|          | 0.00/3.72G [00:00<?, ?B/s]

...

special_tokens_map.json:   0%|          | 0.00/777 [00:00<?, ?B/s]

chat_template.jinja:   0%|          | 0.00/1.63k [00:00<?, ?B/s]

adapter_config.json:   0%|          | 0.00/1.83k [00:00<?, ?B/s]

adapter_model.safetensors:   0%|          | 0.00/76.9M [00:00<?, ?B/s]

trainable params: 19,210,240 || all params: 7,869,188,432 || trainable%: 0.2441


### Load dataset

In [ ]:
from datasets import load_dataset
from unsloth.chat_templates import standardize_data_formats

# Load Spider dataset from Hugging Face
raw_datasets = load_dataset("xlangai/spider", "spider")

# Use the same format standardization function
train_dataset = standardize_data_formats(raw_datasets["train"])
eval_dataset = standardize_data_formats(raw_datasets["validation"])


# We convert each pair into a multi‑turn chat format expected by Unsloth’s chat templates.
TURN_FMT = "<start_of_turn>{role}\n{content}<end_of_turn>\n"

def to_chat(user_text, model_text):
    # Package a (question, sql) pair into the chat turn format.
    return (
        TURN_FMT.format(role='user',  content=user_text) +
        TURN_FMT.format(role='model', content=model_text)
    )

# Adjust field names if your dataset uses different keys
question_key = 'question' if 'question' in train_dataset.column_names else 'question'
sql_key      = 'query'   if 'query'   in train_dataset.column_names else 'query'


# Map each record to a single string of concatenated chat turns
def formatting_prompts_func(examples):
    queries = examples[question_key]
    answers = examples[sql_key]
    texts = [to_chat(q, a) for q, a in zip(queries, answers)]
    return {'text': texts}

# Apply the formatting function to both training and evaluation datasets
train_dataset = train_dataset.map(formatting_prompts_func, batched=True)
eval_dataset  = eval_dataset.map(formatting_prompts_func, batched=True)

# Display one formatted example
print(train_dataset[0]["text"])
print(eval_dataset[0]["text"])

README.md: 0.00B [00:00, ?B/s]

spider/train-00000-of-00001.parquet:   0%|          | 0.00/831k [00:00<?, ?B/s]

spider/validation-00000-of-00001.parquet:   0%|          | 0.00/126k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/7000 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/1034 [00:00<?, ? examples/s]

Map:   0%|          | 0/7000 [00:00<?, ? examples/s]

Map:   0%|          | 0/1034 [00:00<?, ? examples/s]

<start_of_turn>user
How many heads of the departments are older than 56 ?<end_of_turn>
<start_of_turn>model
SELECT count(*) FROM head WHERE age  >  56<end_of_turn>

<start_of_turn>user
How many singers do we have?<end_of_turn>
<start_of_turn>model
SELECT count(*) FROM singer<end_of_turn>



### Connect sql database

In [ ]:
from google.colab import drive
import sqlite3
import os
import pandas as pd
import re
import json

#download add data to your drive

#mount drive
drive.mount('/content/drive', force_remount=True)

# Default paths
BASE_DIR = "/content/drive/MyDrive/Independent_study/spider/database"
TEST_BASE_DIR = "/content/drive/MyDrive/Independent_study/spider/test_database"
TABLES_JSON = "/content/drive/MyDrive/Independent_study/spider/tables.json"
TEST_TABLES_JSON = "/content/drive/MyDrive/Independent_study/spider/test_tables.json"

def run_spider_sql_df(
    sql: str,
    db_id: str,
    mode: str = "train",   # "train" or "test"
    base_dir: str = BASE_DIR,
    test_base_dir: str = TEST_BASE_DIR,
    limit: int | None = None
) -> tuple[pd.DataFrame, str | None]:
    """
    Run SQL on either train or test Spider database and return (DataFrame, error_message).

    Returns
    -------
    df : pd.DataFrame
        DataFrame of query results (empty if error)
    error_message : str | None
        None if success, otherwise contains error message text
    """
    # fix tuple
    if isinstance(sql, tuple):
      sql = sql[0]

    # Select directory depending on mode
    db_root = base_dir if mode == "train" else test_base_dir
    db_path = os.path.join(db_root, db_id, f"{db_id}.sqlite")

    if not os.path.exists(db_path):
        return pd.DataFrame(), f"❌ Database not found: {db_path}"

    conn = sqlite3.connect(db_path)
    conn.execute("PRAGMA foreign_keys = ON;")

    q = sql.strip()
    q = re.sub(r";\s*$", "", q)

    if limit is not None and "limit" not in q.lower():
        q = f"SELECT * FROM ({q}) AS _sub LIMIT {int(limit)}"

    try:
        df = pd.read_sql_query(q, conn)
        conn.close()
        return df, None
    except Exception as e:
        conn.close()
        error_message = str(e)
        return pd.DataFrame(), error_message

def get_schema_from_tables_json(
    db_id: str,
    mode: str = "train",   # "train" or "test"
    tables_json_path: str = TABLES_JSON,
    test_tables_json_path: str = TEST_TABLES_JSON,
    base_dir: str = BASE_DIR,
    test_base_dir: str = TEST_BASE_DIR,
    as_string: bool = True
):
    """
    Extract schema (columns, PK, FK) for given db_id from either tables.json or test_tables.json.
    """

    # Auto-select JSON & base folder
    if mode == "test":
        tables_json_path = test_tables_json_path
        base_dir = test_base_dir

    if not os.path.exists(tables_json_path):
        raise FileNotFoundError(f"tables.json not found at {tables_json_path}")

    with open(tables_json_path, "r", encoding="utf-8") as f:
        tables_data = json.load(f)

    db_entry = None
    for db in tables_data["dbs"] if "dbs" in tables_data else tables_data:
        if db["db_id"] == db_id:
            db_entry = db
            break

    if db_entry is None:
        raise ValueError(f"db_id '{db_id}' not found in {tables_json_path}")

    table_names = db_entry["table_names_original"]
    columns = db_entry["column_names_original"]
    column_types = db_entry["column_types"]
    foreign_keys = db_entry["foreign_keys"]
    primary_keys = db_entry["primary_keys"]

    schema_dict = {t: {"columns": [], "foreign_keys": [], "primary_keys": []} for t in table_names}

    for (table_id, col_name), col_type in zip(columns, column_types):
        if table_id == -1:
            continue
        table_name = table_names[table_id]
        schema_dict[table_name]["columns"].append(f"{col_name} ({col_type})")

    for pk_idx in primary_keys:
        t_idx, col_name = columns[pk_idx]
        if t_idx >= 0:
            schema_dict[table_names[t_idx]]["primary_keys"].append(col_name)

    for (from_col_idx, to_col_idx) in foreign_keys:
        from_t_idx, from_col = columns[from_col_idx]
        to_t_idx, to_col = columns[to_col_idx]
        if from_t_idx >= 0 and to_t_idx >= 0:
            schema_dict[table_names[from_t_idx]]["foreign_keys"].append(
                f"{from_col} -> {table_names[to_t_idx]}.{to_col}"
            )

    # Optional SQLite verification
    db_path = os.path.join(base_dir, db_id, f"{db_id}.sqlite")
    if not os.path.exists(db_path):
        print(f"⚠️ SQLite file not found for {db_id}, returning only JSON schema.")
    else:
        try:
            conn = sqlite3.connect(db_path)
            for table in schema_dict.keys():
                _ = conn.execute(f"PRAGMA table_info({table});").fetchall()
            conn.close()
        except Exception as e:
            print(f"⚠️ SQLite check failed: {e}")

    if as_string:
        lines = []
        for t, info in schema_dict.items():
            lines.append(f"Table {t}, columns = [{', '.join(info['columns'])}]")
            if info["primary_keys"]:
                lines.append(f"Primary Key: {', '.join(info['primary_keys'])}")
            if info["foreign_keys"]:
                lines.append(f"Foreign Keys: {', '.join(info['foreign_keys'])}")
        return "\n".join(lines)
    else:
        return schema_dict

def get_all_tables_from_db(
    db_id: str,
    mode: str = "train",   # "train" or "test"
    base_dir: str = BASE_DIR,
    test_base_dir: str = TEST_BASE_DIR
) -> list[str]:
    """
    Return all table names in a given Spider database (train/test mode).

    Parameters
    ----------
    db_id : str
        Database ID (folder name)
    mode : str
        Either "train" or "test"
    base_dir : str
        Path to 'database' directory for train
    test_base_dir : str
        Path to 'test_database' directory for test

    Returns
    -------
    list[str]
        List of table names available in the specified SQLite database
    """

    # Select directory depending on mode
    db_root = base_dir if mode == "train" else test_base_dir
    db_path = os.path.join(db_root, db_id, f"{db_id}.sqlite")

    if not os.path.exists(db_path):
        raise FileNotFoundError(f"❌ Database not found: {db_path}")

    conn = sqlite3.connect(db_path)
    cursor = conn.cursor()

    try:
        # Query SQLite system table for all table names
        cursor.execute("SELECT name FROM sqlite_master WHERE type='table';")
        tables = [row[0] for row in cursor.fetchall()]
    finally:
        conn.close()

    return tables

def sample_rows_for_tables(db_id: str, mode: str, tables: list, per_table: int = 3) -> str:
    """
    ดึงตัวอย่างแถวเล็กน้อยจากตารางที่คาดว่าเกี่ยวข้อง เพื่อให้โมเดลเห็นค่าจริง ๆ (ช่วยแก้ empty)
    """
    BASE_DIR = "/content/drive/MyDrive/Independent_study/spider/database"
    TEST_BASE_DIR = "/content/drive/MyDrive/Independent_study/spider/test_database"
    base_dir = BASE_DIR
    test_base_dir = TEST_BASE_DIR
    db_root = base_dir if mode == "train" else test_base_dir
    db_path = os.path.join(db_root, db_id, f"{db_id}.sqlite")
    if not os.path.exists(db_path) or not tables:
        return "(no samples)"
    conn = sqlite3.connect(db_path)
    blocks = []
    for t in tables[:6]:
        try:
            df = pd.read_sql_query(f"SELECT * FROM \"{t}\" LIMIT {int(per_table)};", conn)
            blocks.append(f"Table {t} (sample):\n{df.to_string(index=False)}")
        except Exception:
            continue
    conn.close()
    return "\n\n".join(blocks) if blocks else "(no samples)"

Mounted at /content/drive


In [ ]:
# For normal (train) database
df = run_spider_sql_df("SELECT * FROM institution;", db_id="conference", mode="test")
display(df)

(   Institution_ID              Institution_Name                Location  \
 0               1     Illinois State University        Normal, Illinois   
 1               2            Bradley University        Peoria, Illinois   
 2               3                Eureka College        Eureka, Illinois   
 3               4     Hedding College (defunct)      Abingdon, Illinois   
 4               5              Illinois College  Jacksonville, Illinois   
 5               6  Illinois Wesleyan University   Bloomington, Illinois   
 6               7     Lincoln College, Illinois       Lincoln, Illinois   
 7               8     Lombard College (defunct)     Galesburg, Illinois   
 8               9           Millikin University       Decatur, Illinois   
 9              10   Shurtleff College (defunct)         Alton, Illinois   
 
    Founded  
 0     1857  
 1     1897  
 2     1855  
 3     1855  
 4     1829  
 5     1850  
 6     1865  
 7     1853  
 8     1901  
 9     1827  ,
 None)

In [ ]:
schema_text = get_schema_from_tables_json("conference", mode="test")
print(schema_text)

Table conference, columns = [Conference_ID (number), Conference_Name (text), Year (number), Location (text)]
Primary Key: Conference_ID
Table institution, columns = [Institution_ID (number), Institution_Name (text), Location (text), Founded (number)]
Primary Key: Institution_ID
Table staff, columns = [staff_ID (number), name (text), Age (number), Nationality (text), Institution_ID (number)]
Primary Key: staff_ID
Foreign Keys: Institution_ID -> institution.Institution_ID
Table conference_participation, columns = [Conference_ID (number), staff_ID (number), role (text)]
Primary Key: staff_ID
Foreign Keys: Conference_ID -> conference.Conference_ID, staff_ID -> staff.staff_ID


### Create model/Skip this if loaded finetuned

In [ ]:
model = FastModel.get_peft_model(
    model,
    finetune_vision_layers     = False, # Turn off for just text!
    finetune_language_layers   = True,  # Should leave on!
    finetune_attention_modules = True,  # Attention good for GRPO
    finetune_mlp_modules       = True,  # SHould leave on always!

    r = 8,           # Larger = higher accuracy, but might overfit
    lora_alpha = 8,  # Recommended alpha == r at least
    lora_dropout = 0,
    bias = "none",
    random_state = 5512,
)

RuntimeError: Unsloth: You already added LoRA adapters to your model!

In [ ]:
train_dataset = train_dataset.shuffle(seed=42)

In [ ]:
train_dataset[0:10]['db_id']

In [ ]:
from trl import SFTTrainer, SFTConfig
import datetime

timestamp = datetime.datetime.now().strftime("%Y-%m-%d_%H-%M")

arg = SFTConfig(
    dataset_text_field = "text",
    per_device_train_batch_size = 1,
    gradient_accumulation_steps = 4, # Use GA to mimic batch size!
    warmup_steps = 10,
    num_train_epochs = 1, # Set this for 1 full training run.
    # max_steps = 1000,
    learning_rate = 2e-5, # Reduce to 2e-5 for long training runs
    logging_steps = 1,
    optim = "adamw_8bit",
    weight_decay = 0.01,
    lr_scheduler_type = "linear",
    seed = 5512,
    report_to = "none", # Use this for WandB etc

    # == Push to Hub ==
    push_to_hub = True,
    hub_model_id = "suparuek2405/spider-gemma3n-e4b-sft_1epoch",  # ตั้งชื่อ repo
    hub_private_repo = True,       # ทำเป็น private ได้
    hub_strategy = "every_save",   # push ทุกครั้งที่ save
    # hub_strategy = "checkpoint", # หรือ push เฉพาะ checkpoint-* เท่านั้น
    # hub_strategy = "end",        # หรือ push เมื่อจบการฝึก
)

trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = train_dataset,
    eval_dataset = None,
    args = arg,
    formatting_func = formatting_prompts_func,
    )

#train on response only
from unsloth.chat_templates import train_on_responses_only
trainer = train_on_responses_only(
    trainer,
    instruction_part = "<start_of_turn>user\n",
    response_part = "<start_of_turn>model\n",
)

In [ ]:
# title Show current memory stats
gpu_stats = torch.cuda.get_device_properties(0)
start_gpu_memory = round(torch.cuda.max_memory_reserved() / 1024 / 1024 / 1024, 3)
max_memory = round(gpu_stats.total_memory / 1024 / 1024 / 1024, 3)
print(f"GPU = {gpu_stats.name}. Max memory = {max_memory} GB.")
print(f"{start_gpu_memory} GB of memory reserved.")

GPU = NVIDIA L4. Max memory = 22.161 GB.
9.307 GB of memory reserved.


### Start training

#### train new one

In [ ]:
# To resume a training run, set trainer.train(resume_from_checkpoint = True)
trainer_stats = trainer.train()

trainer.push_to_hub(commit_message=f"Final save at {timestamp}")

==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 7,000 | Num Epochs = 1 | Total steps = 1,750
O^O/ \_/ \    Batch size per device = 1 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (1 x 4 x 1) = 4
 "-____-"     Trainable parameters = 19,210,240 of 7,869,188,432 (0.24% trained)


Step,Training Loss
1,12.323900
2,11.924000
3,10.041500
4,5.046500
5,5.148300
6,3.458800
7,2.988400
8,3.191000
9,3.125000
10,2.847900


Unsloth: Will smartly offload gradients to save VRAM!


Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...eckpoints/tokenizer.model: 100%|##########| 4.70MB / 4.70MB            

  ...heckpoints/tokenizer.json: 100%|##########| 33.4MB / 33.4MB            

  ...adapter_model.safetensors:   0%|          | 61.4kB / 76.9MB            

  ...kpoints/training_args.bin:   6%|5         |   364B / 6.29kB            

CommitInfo(commit_url='https://huggingface.co/suparuek2405/spider-gemma3n-e4b-sft_1epoch/commit/b9e92c02a8476f39ced4ad83e54156a9a703ad11', commit_message='Final save at 2025-11-30_07-40', commit_description='', oid='b9e92c02a8476f39ced4ad83e54156a9a703ad11', pr_url=None, repo_url=RepoUrl('https://huggingface.co/suparuek2405/spider-gemma3n-e4b-sft_1epoch', endpoint='https://huggingface.co', repo_type='model', repo_id='suparuek2405/spider-gemma3n-e4b-sft_1epoch'), pr_revision=None, pr_num=None)

In [ ]:
# title Show final memory and time stats
used_memory = round(torch.cuda.max_memory_reserved() / 1024 / 1024 / 1024, 3)
used_memory_for_lora = round(used_memory - start_gpu_memory, 3)
used_percentage = round(used_memory / max_memory * 100, 3)
lora_percentage = round(used_memory_for_lora / max_memory * 100, 3)
print(f"{trainer_stats.metrics['train_runtime']} seconds used for training.")
print(
    f"{round(trainer_stats.metrics['train_runtime']/60, 2)} minutes used for training."
)
print(f"Peak reserved memory = {used_memory} GB.")
print(f"Peak reserved memory for training = {used_memory_for_lora} GB.")
print(f"Peak reserved memory % of max memory = {used_percentage} %.")
print(f"Peak reserved memory for training % of max memory = {lora_percentage} %.")

8423.9384 seconds used for training.
140.4 minutes used for training.
Peak reserved memory = 9.768 GB.
Peak reserved memory for training = 0.461 GB.
Peak reserved memory % of max memory = 44.077 %.
Peak reserved memory for training % of max memory = 2.08 %.


### Inference

In [ ]:
print("Model max position embeddings (text):", model.config.text_config.max_position_embeddings)
print("Sliding window (per sliding layer):", model.config.text_config.sliding_window)
# print("Tokenizer model_max_length:", tokenizer.model_max_length)

Model max position embeddings (text): 32768
Sliding window (per sliding layer): 512


In [ ]:
# count token
from transformers import AutoTokenizer

def count_tokens_hf(text: str, model_id: str =  "unsloth/gemma-3n-E4B-it") -> int:
    """
    นับจำนวน token จาก tokenizer ของโมเดล HuggingFace เช่น Gemma, LLaMA, Mistral
    """
    tokenizer = AutoTokenizer.from_pretrained(model_id)
    count = len(tokenizer.encode(text))
    print(f"Token count: {count}")
    # return count

# ตัวอย่าง
prompt = "List the top 5 airports by number of departing flights in 2017."
count_tokens_hf(prompt)

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.model:   0%|          | 0.00/4.70M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/33.4M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/777 [00:00<?, ?B/s]

chat_template.jinja: 0.00B [00:00, ?B/s]

Token count: 19


In [ ]:
# Generate response text from a list of chat messages.
def get_response_text(messages, model, tokenizer, max_new_tokens=2048, temperature=0.7, top_p=0.95, top_k=50):
    # Prepare input tensors
    inputs = tokenizer.apply_chat_template(
        messages,
        add_generation_prompt=True,
        return_tensors='pt',
        tokenize=True,
        return_dict=True,
    ).to(model.device)

    # Generate output ids
    output_ids = model.generate(
        **inputs,
        max_new_tokens=max_new_tokens,
        temperature=temperature,
        top_p=top_p,
        top_k=top_k,
        do_sample=True,
        eos_token_id=tokenizer.eos_token_id,
    )

    # Extract only the generated portion after the prompt
    generated_ids = output_ids[0][inputs['input_ids'].shape[-1]:]
    return tokenizer.decode(generated_ids, skip_special_tokens=True).strip()


#### Schema linking module

In [ ]:
schema_text = get_schema_from_tables_json("employee_hire_evaluation", mode="test")
print(schema_text)

Table employee, columns = [Employee_ID (number), Name (text), Age (number), City (text)]
Primary Key: Employee_ID
Table shop, columns = [Shop_ID (number), Name (text), Location (text), District (text), Number_products (number), Manager_name (text)]
Primary Key: Shop_ID
Table hiring, columns = [Shop_ID (number), Employee_ID (number), Start_from (text), Is_full_time (others)]
Primary Key: Employee_ID
Foreign Keys: Employee_ID -> employee.Employee_ID, Shop_ID -> shop.Shop_ID
Table evaluation, columns = [Employee_ID (text), Year_awarded (text), Bonus (number)]
Primary Key: Employee_ID
Foreign Keys: Employee_ID -> employee.Employee_ID


In [ ]:
def schema_linking_module(question, db_id, mode, schema):
    system_prompt = """You are a schema-linking assistant for the Spider Text-to-SQL task.

Goal:
Given a database schema and a natural-language question,
identify ONLY the minimal set of schema elements and literal values required to build the correct SQL.

Output format (single line only):
Schema_links: [table.column, table.column, ..., literal1, literal2, ...]

Guidelines:
1. Use EXACT column names from the provided schema (case-sensitive).
2. Include ONLY columns or tables essential for the SQL — omit anything irrelevant.
3. Always include columns explicitly or implicitly mentioned in the question
   (e.g., name, id, date, category, status, year, amount).
4. For aggregation, comparison, or ordering questions, include:
   - measure columns (population, salary, score, etc.)
   - grouping or filter columns
   - ranking columns for superlatives (largest, newest, highest, etc.)
5. For JOINs, include all join keys and show at least one shortest FK path:
   FK: table1.col -> table2.col
6. Extract literal values (names, numbers, years, dates, strings) exactly as stated in the question.
   - Example: 'Katie White', 2012, 'USA'
   - Keep relative time phrases literal (e.g., 'last 7 days', 'before 2010').
7. Do NOT invent columns, tables, or aliases not in the schema.
8. Do NOT output SQL or explanations.
9. When multiple columns could match, prefer IDs or main textual columns (e.g., name over description).
10. For counting, include a stable key column (e.g., table.id) of the counted entity.

Return only a single valid line starting with 'Schema_links:'.
"""

    # schema = get_schema_from_tables_json(db_id, mode=mode)
    final_prompt = (
        system_prompt
        + "\n\nDatabase schema:\n"
        + schema
        + "\n\nQuestion: "
        + question
        + "\nAnswer:"
    )

    messages = [{"role": "user", "content": [{"type": "text", "text": final_prompt}]}]
    response = get_response_text(messages, model, tokenizer, max_new_tokens=1024)

    # return final_prompt
    # print(final_prompt)
    return response

In [ ]:
# print(schema_linking_module("Find the districts in which there are both shops selling less than 3000 products and shops selling more than 10000 products.","employee_hire_evaluation", "test"))

#### SQL generator module

In [ ]:
def sql_generator_module(question, db_id, mode, schema_links):
  system_prompt = """You are a professional Text-to-SQL model trained on the Spider dataset.

Your task:
Generate **only valid SQL statements** (in SQLite syntax) based on the natural language question,
the provided database schema, and the relevant schema links.

Strict rules:
1. Output only raw SQL — no explanations, no reasoning steps, no markdown, no comments.
2. Follow correct SQL syntax for SQLite.
3. Use correct table and column names exactly as provided in the schema.
4. If the question involves aggregation (sum, count, avg, min, max), use DISTINCT where needed.
5. If filtering by values, use the literal values exactly as they appear (e.g., 'USA', 2010).
6. If schema_links are provided, limit your SQL to those relevant tables and columns.
7. If the question implies ordering (e.g., "highest", "latest", "most recent"), include ORDER BY.
8. If it asks for limits (e.g., "top 5", "first 10"), include LIMIT accordingly.
9. Do not include natural language explanations or markdown formatting.
10. Return only one complete SQL statement.
"""
  schema = get_schema_from_tables_json(db_id, mode=mode)
  # schema_links = schema_linking_module(question, db_id, mode)
  final_prompt = (
        system_prompt
        + "\nDatabase schema:\n"
        + schema + "\n"
        + "\n" + schema_links
        + "\n\nQuestion: "
        + question
        + "\nAnswer:"
    )

  messages = [{"role": "user", "content": [{"type": "text", "text": final_prompt}]}]
  response = get_response_text(messages, model, tokenizer, max_new_tokens=1024)

  # return final_prompt
  # print(final_prompt)
  return response

In [ ]:
# print(sql_generator_module("Which employee received the most awards in evaluations? Give me the employee name.","employee_hire_evaluation", "test"))

#### Self correction module

In [ ]:
def self_correction_module(question, db_id, mode, schema, schema_link, sql, df, er):
  ok_syntax = er is None
  non_empty = (df is not None) and (len(df) > 0)

  if ok_syntax and non_empty:
    return sql

  # when df empty
  if not ok_syntax or not non_empty:
    if er is None:
      er = 'This SQL return empty result'
    system_prompt = """You are a SQL Repair Assistant specialized for the Spider Text-to-SQL benchmark.
    The following SQL may have some errors. Correct any mistakes to ensure it answers the question correctly.

Your goal:
Given the following inputs, generate a corrected SQL query that both EXECUTES successfully
and RETURNS meaningful (non-empty, logically consistent) results for the given question.

Inputs:
1. Database SCHEMA (with primary/foreign keys)
2. Schema_links (relevant columns/tables)
3. QUESTION (natural language)
4. Data samples (for understanding column semantics only — do not fabricate values)
5. Previous SQL (the failed or empty-result query)
6. Error message (from the previous SQL execution)

Strict Rules:
- Output ONLY the SQL query. Do NOT include explanations or commentary.
- Use ONLY existing tables/columns as shown in the schema (case-sensitive).
- Prefer explicit joins using provided foreign keys.
- Avoid over-restrictive WHERE conditions; relax filters if necessary (use LIKE or broader date ranges).
- Keep SELECT columns relevant to the QUESTION intent.
- If aggregation, counting, or ranking is implied, include COUNT, ORDER BY, or LIMIT appropriately.
- If prior query returned an empty result, ensure your SQL broadens the search scope slightly.
- Ensure syntax correctness (runnable SQL for SQLite).

Now use this information to produce a single corrected SQL query:
"""
  tb = get_all_tables_from_db(db_id, mode)
  samples = sample_rows_for_tables(db_id, mode, tb, per_table=3)
  final_prompt = (
        system_prompt
        + "\nDatabase schema:\n"
        + schema + "\n"
        + "\n" + schema_link
        + "\n\nQuestion: "
        + question + "\n"
        + "\n" + "Data samples (for guidance only, do not invent values):"
        + "\n" + samples + "\n"
        + "\n" + "Previous SQL: " + sql + "\n"
        + "\n"+ "Error message: " + er + "\n"
        + "\n" + "Use ONLY existing tables/columns as shown in the schema (case-sensitive)."
        + "\nAnswer:"
    )

  messages = [{"role": "user", "content": [{"type": "text", "text": final_prompt}]}]
  response = get_response_text(messages, model, tokenizer, max_new_tokens=1024)

  # print(final_prompt)
  return response

#### Combine all module

#### sql generator only

In [ ]:
import time

def gemma_sql_01(question, db_id, mode, max_rounds=3):
    schema = get_schema_from_tables_json(db_id, mode)
    # schema_link = schema_linking_module(question, db_id, mode, schema)
    schema_link = ''
    sql = sql_generator_module(question, db_id, mode, schema_link)
    df, er = run_spider_sql_df(sql, db_id=db_id, mode=mode)

    new_sql = sql
    results_log = []
    return new_sql, df, results_log

#### sql generator + schema linking

In [ ]:
import time

def gemma_sql_02(question, db_id, mode, max_rounds=3):
    schema = get_schema_from_tables_json(db_id, mode)
    schema_link = schema_linking_module(question, db_id, mode, schema)
    sql = sql_generator_module(question, db_id, mode, schema_link)
    df, er = run_spider_sql_df(sql, db_id=db_id, mode=mode)

    new_sql = sql
    results_log = []

    return new_sql, df, results_log

#### gemmasql - all module

In [ ]:
import time

def gemma_sql(question, db_id, mode, max_rounds=3):
    schema = get_schema_from_tables_json(db_id, mode)
    schema_link = schema_linking_module(question, db_id, mode, schema)
    sql = sql_generator_module(question, db_id, mode, schema_link)
    df, er = run_spider_sql_df(sql, db_id=db_id, mode=mode)

    rnd_fix = 0
    results_log = []  # optional log per round

    for rnd in range(max_rounds):
        rnd_fix += 1
        # print(f"\n===== Round {rnd_fix} =====")
        start_time = time.time()  # ⏱ start timer

        # Try correction
        new_sql = self_correction_module(question, db_id, mode, schema, schema_link, sql, df, er)
        df, er = run_spider_sql_df(new_sql, db_id=db_id, mode=mode)

        end_time = time.time()  # ⏱ end timer
        duration = end_time - start_time

        ok_syntax = er is None
        non_empty = (df is not None) and (len(df) > 0)

        # --- logging and printing runtime ---
        # print(f"⏱ Runtime: {duration:.2f} seconds")
        # print(f"SQL: {new_sql[:150]}{'...' if len(new_sql) > 150 else ''}")

        # Record the results
        results_log.append({
            "round": rnd_fix,
            "sql": new_sql,
            "rows": len(df) if df is not None else 0,
            "error": er,
            "ok_syntax": ok_syntax,
            "non_empty": non_empty,
            "runtime_sec": duration
        })

        # --- Conditions ---
        if ok_syntax and non_empty:
            # print(f"✅ Success on Round {rnd_fix}")
            return new_sql, df, results_log

        if not ok_syntax:
            # print(f"❌ Syntax error on Round {rnd_fix}: {er}")
            continue

        if not non_empty:
            # print(f"⚠️ Empty result on Round {rnd_fix}")
            continue

        if rnd_fix == max_rounds:
          pass
            # print(f"🕒 Reached final round ({rnd_fix}) without success")

    # print("❗ All rounds finished without successful query")
    return new_sql, df, results_log

In [ ]:
question = "Which shops number products is above the average? Give me the shop names."
db_id = "employee_hire_evaluation"
mode = "test"

# schema = get_schema_from_tables_json(db_id, mode)
# schema_link = schema_linking_module(question, db_id, mode, schema)
# sql = sql_generator_module(question, db_id, mode, schema_link)
# df, er = run_spider_sql_df(sql, db_id=db_id, mode=mode)

sql,df,log = gemma_sql(question, db_id, mode, max_rounds=10)

In [ ]:
sql

'SELECT Name FROM shop WHERE Number_products > (SELECT avg(Number_products) FROM shop);'

### Evaluation

#### Exact match

In [ ]:
import re
import unicodedata

def normalize_sql(sql: str | None) -> str | None:
    if sql is None:
        return None

    # Normalize unicode (remove zero-width spaces, NBSP, etc.)
    sql = unicodedata.normalize("NFKC", sql)
    sql = sql.replace("\u200b", "")   # zero-width space
    sql = sql.replace("\u00A0", " ")  # non-breaking space
    sql = sql.replace("\t", " ")      # tabs → space

    # Lowercase + strip
    s = sql.strip().lower()

    # Collapse any run of whitespace to a single space
    s = re.sub(r"\s+", " ", s)

    # 🔧 NEW: normalize spaces around commas
    # any spaces before/after comma → exactly ", "
    s = re.sub(r"\s*,\s*", ", ", s)

    # (Optional) normalize spaces just inside parentheses: "( a , b )" → "(a, b)"
    s = re.sub(r"\(\s+", "(", s)
    s = re.sub(r"\s+\)", ")", s)

    # Remove backticks / normalize quotes
    s = s.replace("`", "").replace('"', "'")

    # Remove trailing semicolon
    s = s.rstrip(";")

    return s

def exact_match(gold_sql: str | None, pred_sql: str | None) -> int:
    """
    Return 1 if predicted SQL exactly matches gold SQL after normalization, else 0.
    This is a *string-based* EM (not the official Spider component EM).
    """
    gold_norm = normalize_sql(gold_sql)
    pred_norm = normalize_sql(pred_sql)

    if gold_norm is None or pred_norm is None:
        return 0

    return int(gold_norm == pred_norm)

#### Execution accuracy

In [ ]:
import pandas as pd

def _normalize_result_df_ignore_colnames(df: pd.DataFrame | None) -> pd.DataFrame | None:
    """
    สร้าง canonical form ของตาราง เพื่อเทียบเฉพาะ 'ค่าข้อมูล' โดย:
    - ไม่สนชื่อ column
    - ไม่สนลำดับ column
    - ไม่สนลำดับ row
    """
    if df is None:
        return None

    # 1) dtype เป็น string ทั้งหมด เพื่อกันปัญหา int/float/bool ต่างกัน
    df_norm = df.astype(str).copy()

    n_rows, n_cols = df_norm.shape

    # 2) จัดลำดับ column ใหม่ โดยใช้ vector ของค่าทั้งคอลัมน์เป็น key
    #    ใช้ index (0..n_cols-1) แทนชื่อ column เพื่อไม่ให้ชื่อมีผล
    col_order = sorted(
        range(n_cols),
        key=lambda j: tuple(df_norm.iloc[:, j].tolist())
    )
    df_norm = df_norm.iloc[:, col_order]

    # 3) รีเซ็ตชื่อ column เป็น 0..n_cols-1 ให้เหมือนกันทุก DataFrame
    df_norm.columns = list(range(n_cols))

    # 4) จัดลำดับ row ใหม่ โดย sort ตามทุกคอลัมน์ (ตอนนี้ชื่อเป็นตัวเลขเหมือนกันแล้ว)
    df_norm = df_norm.sort_values(by=list(df_norm.columns), kind="mergesort").reset_index(drop=True)

    return df_norm


def execution_match(gold_df: pd.DataFrame | None,
                                    pred_df: pd.DataFrame | None) -> int:
    """
    คืน 1 ถ้าตารางเท่ากัน (up to permutation ของ row+column และไม่สนชื่อ column)
    คืน 0 ถ้าไม่เท่ากัน
    """
    gold = _normalize_result_df_ignore_colnames(gold_df)
    pred = _normalize_result_df_ignore_colnames(pred_df)

    # กรณี None
    if gold is None or pred is None:
        return int(gold is None and pred is None)

    # shape ไม่เท่ากัน → ไม่เท่ากันแน่นอน
    if gold.shape != pred.shape:
        return 0

    # เทียบ canonical form
    return int(gold.equals(pred))

In [ ]:
a = pd.DataFrame({"A": [1, 2], "B": [3, 4]})
b = pd.DataFrame({"x": [1, 2], "y": [3, 4]})          # ชื่อคอลัมน์ต่าง แต่ข้อมูลเหมือน
c = pd.DataFrame({"B": [3, 4], "A": [1, 2]})          # column สลับ
d = pd.DataFrame({"B": [4, 3], "A": [2, 1]})          # column+row สลับ
e = pd.DataFrame({"A": [9, 9], "B": [9, 9]})          # ข้อมูลต่าง

print(execution_match(a, b))  # 1
print(execution_match(a, c))  # 1
print(execution_match(a, d))  # 1
print(execution_match(a, e))  # 0

1
1
1
0


#### run evaluation

In [ ]:
def evaluate_text2sql(
    eval_dataset,
    mode: str = "test",
    max_examples: int | None = None,
    verbose: bool = False,
    max_rounds: int = 10,
    experiment = 1
):

    total = 0
    em_correct = 0
    ex_correct = 0

    for i, ex in enumerate(eval_dataset):
        if max_examples is not None and i >= max_examples:
            break

        db = ex["db_id"]
        q = ex["question"]
        qr = ex["query"]

        # Get model prediction

        ### get answer with gold sql
        real_answer, real_error = run_spider_sql_df(sql=qr, db_id=db, mode="test")

        ### get model answer
        if experiment == 1:
          model_sql,df,log = gemma_sql_01(question=q, db_id=db, mode="test", max_rounds=max_rounds)

        elif experiment == 2:
          model_sql,df,log = gemma_sql_02(question=q, db_id=db, mode="test", max_rounds=max_rounds)

        elif experiment == 3:
          model_sql,df,log = gemma_sql(question=q, db_id=db, mode="test", max_rounds=max_rounds)


        # EM
        em = exact_match(qr, model_sql)
        # EX
        ex_score = execution_match(real_answer, df)

        em_correct += em
        ex_correct += ex_score
        total += 1

        if verbose:
            print(f"test {i+1} db={db}")
            print(f"  Q : {q}")
            print(f"  gold_sql : {qr}")
            print(f"  pred_sql : {model_sql}")
            print(f"  EM={em}, EX={ex_score}")
            print("-" * 60)

    em_score = em_correct / total if total > 0 else 0.0
    ex_score = ex_correct / total if total > 0 else 0.0

    print(f"Total examples: {total}")
    print(f"Exact Match (EM): {em_score:.4f}")
    print(f"Execution Accuracy (EX): {ex_score:.4f}")

    return {
        "num_examples": total,
        "EM": em_score,
        "EX": ex_score,
        "em_correct": em_correct,
        "ex_correct": ex_correct,
    }

In [ ]:
import numpy as np

n_samples = len(eval_dataset)  # 1034
rng = np.random.default_rng(11)  # fixed seed for repeatability

indices = rng.choice(n_samples, size=1034, replace=False)
indices = indices.tolist()
print(indices)  # these are your 50 random numbers (0–1033)

# use them to select from the dataset
shuffled = eval_dataset.select(indices)

[393, 230, 214, 690, 907, 827, 771, 831, 369, 366, 312, 625, 572, 586, 616, 573, 938, 578, 412, 933, 810, 864, 555, 856, 959, 813, 28, 323, 748, 535, 782, 773, 515, 530, 428, 368, 467, 1025, 764, 483, 175, 757, 403, 683, 649, 377, 420, 828, 719, 977, 852, 443, 506, 598, 918, 423, 124, 314, 316, 846, 735, 904, 482, 489, 844, 1012, 360, 382, 111, 63, 219, 643, 430, 701, 215, 840, 908, 117, 524, 912, 35, 481, 395, 477, 206, 476, 115, 628, 552, 61, 829, 606, 1002, 991, 114, 665, 468, 67, 56, 459, 832, 290, 582, 858, 98, 510, 120, 21, 94, 640, 431, 752, 331, 711, 823, 843, 790, 9, 645, 691, 374, 361, 354, 310, 429, 505, 849, 692, 821, 59, 730, 783, 670, 881, 85, 706, 850, 733, 252, 232, 246, 166, 781, 867, 419, 624, 641, 192, 31, 981, 1019, 517, 13, 191, 355, 754, 929, 756, 100, 591, 129, 390, 574, 306, 39, 1021, 447, 226, 932, 571, 72, 901, 437, 659, 159, 921, 299, 763, 5, 699, 949, 486, 613, 1033, 77, 3, 409, 888, 282, 200, 163, 720, 457, 772, 501, 421, 662, 1000, 364, 97, 760, 898, 240, 

In [ ]:
random_indice = [938, 412, 872, 703, 586,
                 155, 681, 470, 55, 943,
                 73, 811, 866, 557, 858,
                1019, 958, 851, 448, 629,
                 596, 673, 834, 50, 329,
                 1018, 569, 82, 695, 833,
                 175
                 ]
shuffled = eval_dataset.select(random_indice)
print(len(random_indice))
print(len(list(set(random_indice))))

31
31


In [ ]:
for i in range(31):
    print(shuffled[i])

{'db_id': 'dog_kennels', 'query': 'SELECT T1.treatment_type_description FROM Treatment_types AS T1 JOIN Treatments AS T2 ON T1.treatment_type_code  =  T2.treatment_type_code GROUP BY T1.treatment_type_code ORDER BY sum(cost_of_treatment) ASC LIMIT 1', 'question': 'What is the description of the treatment type that costs the least money in total?', 'query_toks': ['SELECT', 'T1.treatment_type_description', 'FROM', 'Treatment_types', 'AS', 'T1', 'JOIN', 'Treatments', 'AS', 'T2', 'ON', 'T1.treatment_type_code', '=', 'T2.treatment_type_code', 'GROUP', 'BY', 'T1.treatment_type_code', 'ORDER', 'BY', 'sum', '(', 'cost_of_treatment', ')', 'ASC', 'LIMIT', '1'], 'query_toks_no_value': ['select', 't1', '.', 'treatment_type_description', 'from', 'treatment_types', 'as', 't1', 'join', 'treatments', 'as', 't2', 'on', 't1', '.', 'treatment_type_code', '=', 't2', '.', 'treatment_type_code', 'group', 'by', 't1', '.', 'treatment_type_code', 'order', 'by', 'sum', '(', 'cost_of_treatment', ')', 'asc', 'lim

In [ ]:
# run experiment 1 2 3 on base/finetune model
metrics = evaluate_text2sql(
    shuffled,
    mode="test",
    max_examples=31,
    verbose=True,   # set True to see per-sample details
    max_rounds=10,
    experiment=3
)

metrics

test 1 db=dog_kennels
  Q : What is the description of the treatment type that costs the least money in total?
  gold_sql : SELECT T1.treatment_type_description FROM Treatment_types AS T1 JOIN Treatments AS T2 ON T1.treatment_type_code  =  T2.treatment_type_code GROUP BY T1.treatment_type_code ORDER BY sum(cost_of_treatment) ASC LIMIT 1
  pred_sql : SELECT T2.treatment_type_description FROM Treatments AS T1 JOIN Treatment_Types AS T2 ON T1.treatment_type_code = T2.treatment_type_code ORDER BY T2.cost_of_treatment ASC LIMIT 1
  EM=0, EX=0
------------------------------------------------------------
test 2 db=museum_visit
  Q : Find the names of the visitors whose membership level is higher than 4, and order the results by the level from high to low.
  gold_sql : SELECT name FROM visitor WHERE Level_of_membership  >  4 ORDER BY Level_of_membership DESC
  pred_sql : SELECT Name FROM visitor WHERE Level_of_membership > 4 ORDER BY Level_of_membership DESC
  EM=1, EX=1
----------------------

{'num_examples': 31,
 'EM': 0.16129032258064516,
 'EX': 0.6774193548387096,
 'em_correct': 5,
 'ex_correct': 21}

### Playground

In [ ]:
df, er = run_spider_sql_df("SELECT Name FROM teacher WHERE Course_ID IN (SELECT Course_ID FROM course WHERE Course = 'Math')", db_id="employee_hire_evaluation", mode="test")
df

""


In [ ]:
df, er = run_spider_sql_df("SELECT T3.Name FROM course_arrange AS T1 JOIN course AS T2 ON T1.Course_ID  =  T2.Course_ID JOIN teacher AS T3 ON T1.Teacher_ID  =  T3.Teacher_ID WHERE T2.Course  =  'Math'", db_id="employee_hire_evaluation", mode="test")
df

""
